# Единый DGP для DML vs RCT vs IV vs DiD

Бизнес-событие во всех дизайнах одно: **первая успешная оплата ЖКХ в приложении**. Сначала генерируется один структурный мир клиентов и их потенциальные исходы, затем к нему применяются разные механизмы назначения воздействия.

- **RCT / ITT:** эффект случайного промо на PnL.
- **IV / LATE:** эффект первой оплаты для клиентов-комплаеров; промо служит инструментом.
- **DML:** эффект органического начала использования при selection on observables.
- **DiD:** динамика PnL после staggered adoption относительно ещё не начавших / never-treated.

Названия `y`, `d`, `z`, propensity и oracle-полей следуют соглашениям `causalis.dgp`. Сам генератор общий, потому что готовые DGP в пакете создают независимые миры и не позволяют честно сравнивать дизайны на одних клиентах.

In [1]:
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd

from causalis.dgp import _sigmoid

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## Параметры сценария

Базовый сценарий удовлетворяет ключевым предпосылкам. Для stress-тестов меняйте `hidden_confounding`, `did_trend_violation`, `anticipation_strength` и `overlap_strength`.

In [2]:
@dataclass(frozen=True)
class WorldConfig:
    n_clients: int = 20_000
    periods: int = 24
    seed: int = 42
    pre_month: int = 11
    treatment_month: int = 12
    post_month: int = 15
    hidden_confounding: float = 0.0
    did_trend_violation: float = 0.0
    anticipation_strength: float = 0.0
    overlap_strength: float = 1.0


CONFIG = WorldConfig()
asdict(CONFIG)

{'n_clients': 20000,
 'periods': 24,
 'seed': 42,
 'pre_month': 11,
 'treatment_month': 12,
 'post_month': 15,
 'hidden_confounding': 0.0,
 'did_trend_violation': 0.0,
 'anticipation_strength': 0.0,
 'overlap_strength': 1.0}

In [3]:
def generate_causal_world(config: WorldConfig = WorldConfig()) -> dict:
    """Generate one latent client world and RCT, IV, DML and DiD views.

    Oracle columns are for simulation validation only and must not be fed to estimators.
    """
    cfg = config
    if not (0 <= cfg.pre_month < cfg.treatment_month <= cfg.post_month < cfg.periods):
        raise ValueError("Require pre_month < treatment_month <= post_month < periods.")
    rng = np.random.default_rng(cfg.seed)
    n, periods = cfg.n_clients, cfg.periods

    # Stable pre-treatment client characteristics.
    income_z = rng.normal(size=n)
    digital = rng.normal(size=n)
    loyalty = _sigmoid(0.8 * digital + 0.3 * income_z + rng.normal(scale=0.8, size=n))
    risk = _sigmoid(-0.5 * income_z + rng.normal(scale=0.9, size=n))
    region = rng.integers(0, 5, size=n)
    products_before = np.clip(
        rng.poisson(np.exp(-0.1 + 0.25 * digital + 0.15 * income_z)), 0, 10
    )
    u_hidden = rng.normal(size=n)
    region_effect = np.array([-30, -10, 0, 15, 35])[region]

    alpha = (250 + 70 * income_z + 50 * digital + 35 * products_before
             + region_effect + 45 * u_hidden + rng.normal(scale=50, size=n))
    untreated_trend = 2.0 + rng.normal(scale=0.8, size=n)
    tau = (120 + 35 * np.tanh(digital) + 25 * (products_before >= 3)
           - 40 * risk + rng.normal(scale=15, size=n))

    clients = pd.DataFrame({
        "client_id": np.arange(n), "income_z": income_z, "digital": digital,
        "loyalty": loyalty, "risk": risk, "products_before": products_before,
        "region": region,
    })

    # Shared untreated potential outcomes with seasonality, macro shocks and AR(1) noise.
    months = np.arange(periods)
    time_effect = 25 * np.sin(2 * np.pi * months / 12) + np.cumsum(rng.normal(scale=3, size=periods))
    eps = np.empty((n, periods))
    eps[:, 0] = rng.normal(scale=80, size=n)
    for month in range(1, periods):
        eps[:, month] = 0.55 * eps[:, month - 1] + rng.normal(scale=65, size=n)
    outlier_mask = rng.uniform(size=(n, periods)) < 0.01
    eps += outlier_mask * rng.standard_t(df=3, size=(n, periods)) * 250
    y0 = alpha[:, None] + untreated_trend[:, None] * months + time_effect + eps

    horizon = cfg.post_month - cfg.treatment_month
    tau_horizon = tau * (1 - np.exp(-(horizon + 1) / 2))

    # RCT: z is randomized promo assignment; d is actual first utility payment.
    z = rng.binomial(1, 0.5, size=n)
    p_d_z0 = _sigmoid(-3.0 + 0.35 * digital - 0.4 * risk)
    p_d_z1 = _sigmoid(1.0 + 0.35 * digital - 0.4 * risk)
    compliance_rank = rng.uniform(size=n)
    d_z0 = (compliance_rank < p_d_z0).astype(int)
    d_z1 = (compliance_rank < p_d_z1).astype(int)
    d_rct = np.where(z == 1, d_z1, d_z0)
    rct = clients.copy()
    rct["z"], rct["d"] = z, d_rct
    rct["y_pre"] = y0[:, cfg.pre_month]
    rct["y"] = y0[:, cfg.post_month] + d_rct * tau_horizon
    rct["y0_true"] = y0[:, cfg.post_month]
    rct["tau_true"] = tau_horizon
    rct["d_z0_true"], rct["d_z1_true"] = d_z0, d_z1
    rct["complier_true"] = ((d_z1 == 1) & (d_z0 == 0)).astype(int)

    # IV is the same experiment with the canonical y/d/z contract.
    iv = rct.copy()
    iv["iv_first_stage_true"] = d_z1 - d_z0
    iv["iv_reduced_form_true"] = (d_z1 - d_z0) * tau_horizon

    # DML: organic adoption selected nonlinearly on observed X.
    observed_score = cfg.overlap_strength * (
        -0.4 + 0.9 * digital + 0.7 * loyalty - 0.6 * risk
        + 0.35 * digital * loyalty + 0.25 * income_z ** 2
    )
    propensity_true = _sigmoid(observed_score + cfg.hidden_confounding * u_hidden)
    d_dml = rng.binomial(1, propensity_true)
    dml = clients.copy()
    dml["d"] = d_dml
    dml["y_pre"] = y0[:, cfg.pre_month]
    dml["y"] = y0[:, cfg.post_month] + d_dml * tau_horizon
    dml["y0_true"] = y0[:, cfg.post_month]
    dml["propensity_true"] = propensity_true
    dml["tau_true"] = tau_horizon

    # DiD: staggered first payment; trend-based selection creates a controlled violation.
    adoption_score = (-0.7 + 0.8 * loyalty + 0.35 * digital
                      + cfg.did_trend_violation * untreated_trend)
    ever_treated = rng.binomial(1, _sigmoid(adoption_score))
    timing_index = adoption_score + rng.normal(scale=0.8, size=n)
    adoption_month = np.full(n, periods + 99, dtype=int)
    cohort = np.select(
        [timing_index > 0.8, timing_index > 0.2, timing_index > -0.4],
        [12, 14, 16], default=18,
    )
    adoption_month[ever_treated == 1] = cohort[ever_treated == 1]
    ids_long, months_long = np.repeat(np.arange(n), periods), np.tile(months, n)
    g_long = np.repeat(adoption_month, periods)
    event_time = months_long - g_long
    treated = ((months_long >= g_long) & (g_long < periods)).astype(int)
    multiplier = np.where(event_time >= 0, 1 - np.exp(-(event_time + 1) / 2), 0.0)
    tau_dynamic = np.repeat(tau, periods) * multiplier
    anticipation = np.where(
        event_time == -1, cfg.anticipation_strength * np.repeat(tau, periods), 0.0
    )
    did = clients.iloc[ids_long].reset_index(drop=True)
    did["month"] = months_long
    did["adoption_month"] = np.where(g_long < periods, g_long, np.nan)
    did["event_time"] = np.where(g_long < periods, event_time, np.nan)
    did["d"] = treated
    did["y"] = y0.reshape(-1) + tau_dynamic + anticipation
    did["y0_true"] = y0.reshape(-1)
    did["tau_true"] = tau_dynamic + anticipation

    complier = (d_z1 == 1) & (d_z0 == 0)
    truth = pd.Series({
        "ate_exposure_horizon": tau_horizon.mean(),
        "att_dml_horizon": tau_horizon[d_dml == 1].mean(),
        "itt_rct_horizon": ((d_z1 - d_z0) * tau_horizon).mean(),
        "late_iv_horizon": tau_horizon[complier].mean(),
        "first_stage_rct": (d_z1 - d_z0).mean(),
    }, name="true_value")
    oracle_columns = {
        "rct": ["y0_true", "tau_true", "d_z0_true", "d_z1_true", "complier_true"],
        "iv": ["y0_true", "tau_true", "d_z0_true", "d_z1_true", "complier_true",
               "iv_first_stage_true", "iv_reduced_form_true"],
        "dml": ["y0_true", "propensity_true", "tau_true"],
        "did": ["y0_true", "tau_true"],
    }
    return {"clients": clients, "rct": rct, "iv": iv, "dml": dml,
            "did": did, "truth": truth, "oracle_columns": oracle_columns,
            "config": asdict(cfg)}

## Генерация датасетов

In [4]:
sim = generate_causal_world(CONFIG)
clients_df = sim["clients"]
rct_df = sim["rct"]
iv_df = sim["iv"]
dml_df = sim["dml"]
did_df = sim["did"]

pd.DataFrame({
    "rows": {k: len(sim[k]) for k in ["clients", "rct", "iv", "dml", "did"]},
    "columns": {k: sim[k].shape[1] for k in ["clients", "rct", "iv", "dml", "did"]},
})

,rows,columns
clients,20000,7
rct,20000,16
iv,20000,18
dml,20000,13
did,480000,14


In [5]:
sim["truth"].to_frame()

,true_value
ate_exposure_horizon,88.224
att_dml_horizon,97.973
itt_rct_horizon,58.048
late_iv_horizon,90.165
first_stage_rct,0.644


## Быстрые sanity checks

Эти проверки не являются оценивателями исследования, но подтверждают механику DGP. Wald ratio должен быть близок к oracle LATE, difference-in-means по `z` — к ITT.

In [6]:
itt_naive = rct_df.groupby("z")["y"].mean().diff().iloc[-1]
first_stage = iv_df.groupby("z")["d"].mean().diff().iloc[-1]
reduced_form = iv_df.groupby("z")["y"].mean().diff().iloc[-1]
wald = reduced_form / first_stage

pd.DataFrame({
    "estimate": [itt_naive, first_stage, wald],
    "oracle": [sim["truth"]["itt_rct_horizon"], sim["truth"]["first_stage_rct"],
               sim["truth"]["late_iv_horizon"]],
}, index=["RCT difference-in-means (ITT)", "IV first stage", "IV Wald (LATE)"])

,estimate,oracle
RCT difference-in-means (ITT),60.720,58.048
IV first stage,0.644,0.644
IV Wald (LATE),94.311,90.165


In [7]:
diagnostics = {
    "rct_assignment_rate": rct_df["z"].mean(),
    "rct_treatment_z0": rct_df.loc[rct_df.z == 0, "d"].mean(),
    "rct_treatment_z1": rct_df.loc[rct_df.z == 1, "d"].mean(),
    "dml_treatment_rate": dml_df["d"].mean(),
    "dml_propensity_p01": dml_df["propensity_true"].quantile(0.01),
    "dml_propensity_p99": dml_df["propensity_true"].quantile(0.99),
    "did_ever_treated_rate": did_df.groupby("client_id")["d"].max().mean(),
}
pd.Series(diagnostics, name="value").to_frame()

,value
rct_assignment_rate,0.493
rct_treatment_z0,0.042
rct_treatment_z1,0.685
dml_treatment_rate,0.488
dml_propensity_p01,0.065
dml_propensity_p99,0.959
did_ever_treated_rate,0.433


In [8]:
did_df.drop_duplicates("client_id")["adoption_month"].fillna("never").value_counts(dropna=False).sort_index(key=lambda s: s.astype(str))

adoption_month
12.000     1342
14.000     1699
16.000     2170
18.000     3455
 never    11334
Name: count, dtype: int64

## Аналитические выборки без oracle leakage

Используйте объекты `*_analysis` в моделях. Полные датафреймы сохраняют истину только для расчёта bias/RMSE/coverage.

In [9]:
rct_analysis = rct_df.drop(columns=sim["oracle_columns"]["rct"])
iv_analysis = iv_df.drop(columns=sim["oracle_columns"]["iv"])
dml_analysis = dml_df.drop(columns=sim["oracle_columns"]["dml"])
did_analysis = did_df.drop(columns=sim["oracle_columns"]["did"])

dml_features = ["income_z", "digital", "loyalty", "risk", "products_before", "region", "y_pre"]
{
    "rct": rct_analysis.columns.tolist(),
    "iv": iv_analysis.columns.tolist(),
    "dml_features": dml_features,
    "did": did_analysis.columns.tolist(),
}

{'rct': ['client_id',
  'income_z',
  'digital',
  'loyalty',
  'risk',
  'products_before',
  'region',
  'z',
  'd',
  'y_pre',
  'y'],
 'iv': ['client_id',
  'income_z',
  'digital',
  'loyalty',
  'risk',
  'products_before',
  'region',
  'z',
  'd',
  'y_pre',
  'y'],
 'dml_features': ['income_z',
  'digital',
  'loyalty',
  'risk',
  'products_before',
  'region',
  'y_pre'],
 'did': ['client_id',
  'income_z',
  'digital',
  'loyalty',
  'risk',
  'products_before',
  'region',
  'month',
  'adoption_month',
  'event_time',
  'd',
  'y']}

## Сценарии нарушений

```python
hidden_world = generate_causal_world(WorldConfig(hidden_confounding=1.0))
bad_did_world = generate_causal_world(WorldConfig(did_trend_violation=1.0))
anticipation_world = generate_causal_world(WorldConfig(anticipation_strength=0.3))
poor_overlap_world = generate_causal_world(WorldConfig(overlap_strength=2.5))
```

При одинаковом `seed` потенциальные исходы и клиентские признаки остаются теми же; меняется только соответствующий механизм назначения/нарушения.

## Опциональный экспорт

Ячейка ниже намеренно не выполняется автоматически. Она сохраняет и аналитические, и oracle-версии рядом с notebook.

In [10]:
EXPORT = False
if EXPORT:
    output_dir = Path("dml_rct_did_iv_data")
    output_dir.mkdir(parents=True, exist_ok=True)
    for name, frame in {
        "clients": clients_df, "rct": rct_analysis, "iv": iv_analysis,
        "dml": dml_analysis, "did": did_analysis,
        "rct_oracle": rct_df, "iv_oracle": iv_df,
        "dml_oracle": dml_df, "did_oracle": did_df,
    }.items():
        frame.to_parquet(output_dir / f"{name}.parquet", index=False)
    sim["truth"].to_frame().to_parquet(output_dir / "truth.parquet")
    print(f"Saved to {output_dir.resolve()}")

# Проверяем

In [11]:
clients_df.head()

,client_id,income_z,digital,loyalty,risk,products_before,region
0,0,0.305,-1.658,0.124,0.635,0,2
1,1,-1.040,0.891,0.353,0.560,3,2
2,2,0.750,-0.058,0.609,0.731,1,0
3,3,0.941,-0.600,0.481,0.201,2,1
4,4,-1.951,-0.251,0.071,0.369,1,1


In [12]:
rct_df.head()

,client_id,income_z,digital,loyalty,risk,products_before,region,z,d,y_pre,y,y0_true,tau_true,d_z0_true,d_z1_true,complier_true
0,0,0.305,-1.658,0.124,0.635,0,2,1,1,105.048,291.477,235.545,55.932,0,1,1
1,1,-1.040,0.891,0.353,0.560,3,2,1,0,357.247,307.036,307.036,107.639,0,0,0
2,2,0.750,-0.058,0.609,0.731,1,0,1,1,253.680,505.302,431.588,73.714,0,1,1
3,3,0.941,-0.600,0.481,0.201,2,1,0,0,401.763,238.932,238.932,89.378,0,1,1
4,4,-1.951,-0.251,0.071,0.369,1,1,1,0,66.070,232.011,232.011,83.864,0,0,0


In [13]:
dml_df.head()

,client_id,income_z,digital,loyalty,risk,products_before,region,d,y_pre,y,y0_true,propensity_true,tau_true
0,0,0.305,-1.658,0.124,0.635,0,2,1,105.048,291.477,235.545,0.097,55.932
1,1,-1.040,0.891,0.353,0.560,3,2,1,357.247,414.675,307.036,0.667,107.639
2,2,0.750,-0.058,0.609,0.731,1,0,1,253.680,505.302,431.588,0.417,73.714
3,3,0.941,-0.600,0.481,0.201,2,1,0,401.763,238.932,238.932,0.354,89.378
4,4,-1.951,-0.251,0.071,0.369,1,1,1,66.070,315.875,232.011,0.537,83.864
